# De la text la imagine

Stable Diffusion desfăcut în componente cu Diffusers și Transformers. Codificatorul de text CLIP, U-Net-ul care lucrează în spațiul latent, autoencoderul VAE și ghidarea fără clasificator scrisă de mână, apoi scala de ghidare și generarea de la imagine la imagine.

Codul din acest notebook vine de pe pagina [De la text la imagine](https://inteligentartificiala.ro/laboratoare/difuzie-imagini-si-sunete/imagini-din-text/) a laboratorului [Imagini și sunete cu modele de difuzie](https://inteligentartificiala.ro/laboratoare/difuzie-imagini-si-sunete/prezentare/), de pe [InteligentArtificiala.ro](https://inteligentartificiala.ro). Explicațiile fiecărui pas sunt pe pagină.

În Google Colab, codul rulează mult mai repede pe o placă grafică. Dacă nu este deja aleasă, se alege din meniul **Runtime**, opțiunea **Change runtime type**, o variantă cu GPU. Celulele se rulează în ordine, de sus în jos.

In [ ]:
%pip install -q -U diffusers transformers accelerate

## Textul devine vectori

Repozitoriul unui model din Diffusers are câte un director pentru fiecare componentă: `tokenizer`, `text_encoder`, `unet`, `vae` și `scheduler`. Fiecare se poate încărca separat, cu argumentul `subfolder`. Codul de mai jos transformă descrierea în vectori cu tokenizatorul și cu codificatorul de text al lui [CLIP](https://inteligentartificiala.ro/modele-de-difuzie/generarea-imaginilor/#cum-ajunge-textul-în-imagine), care vin din biblioteca Transformers.

In [ ]:
import torch
from transformers import CLIPTextModel, CLIPTokenizer

model_id = "stable-diffusion-v1-5/stable-diffusion-v1-5"
tokenizer = CLIPTokenizer.from_pretrained(model_id, subfolder="tokenizer")
text_encoder = CLIPTextModel.from_pretrained(model_id, subfolder="text_encoder", variant="fp16", dtype=torch.float32)

prompt = "a watercolor painting of an orange cat sleeping on a park bench"
tokens = tokenizer(
    prompt, padding="max_length", max_length=tokenizer.model_max_length, truncation=True, return_tensors="pt"
)
print("Primele tokenuri:", tokenizer.convert_ids_to_tokens(tokens.input_ids[0, :15]))
with torch.no_grad():
    embeddings = text_encoder(tokens.input_ids).last_hidden_state
print("Vectorii textului:", tuple(embeddings.shape))

## Generarea, cu ghidarea fără clasificator

In [ ]:
import torch
from diffusers import AutoencoderKL, DDIMScheduler, UNet2DConditionModel
from PIL import Image
from transformers import CLIPTextModel, CLIPTokenizer

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
dtype = torch.float16 if device != "cpu" else torch.float32
model_id = "stable-diffusion-v1-5/stable-diffusion-v1-5"
tokenizer = CLIPTokenizer.from_pretrained(model_id, subfolder="tokenizer")
text_encoder = CLIPTextModel.from_pretrained(model_id, subfolder="text_encoder", variant="fp16", dtype=dtype).to(device)
unet = UNet2DConditionModel.from_pretrained(model_id, subfolder="unet", variant="fp16", dtype=dtype).to(device)
vae = AutoencoderKL.from_pretrained(model_id, subfolder="vae", variant="fp16", dtype=dtype).to(device)
scheduler = DDIMScheduler.from_pretrained(model_id, subfolder="scheduler")


def encode(text):
    ids = tokenizer(
        text, padding="max_length", max_length=tokenizer.model_max_length, truncation=True, return_tensors="pt"
    ).input_ids
    with torch.no_grad():
        return text_encoder(ids.to(device)).last_hidden_state


cond = encode("a watercolor painting of an orange cat sleeping on a park bench")  # c
uncond = encode("")  # ∅, textul gol
w = 7.5  # scala de ghidare

generator = torch.Generator("cpu").manual_seed(0)
latents = torch.randn((1, 4, 64, 64), generator=generator).to(device, dtype)  # zgomot pur, în spațiul latent
scheduler.set_timesteps(50)
for t in scheduler.timesteps:
    with torch.no_grad():
        noise_cond = unet(latents, t, encoder_hidden_states=cond).sample
        noise_uncond = unet(latents, t, encoder_hidden_states=uncond).sample
    noise_pred = noise_uncond + w * (noise_cond - noise_uncond)
    latents = scheduler.step(noise_pred, t, latents).prev_sample

with torch.no_grad():
    image = vae.decode(latents / vae.config.scaling_factor).sample  # din spațiul latent înapoi în pixeli
pixels = ((image[0].clamp(-1, 1) + 1) * 127.5).permute(1, 2, 0).round().byte().cpu().numpy()
Image.fromarray(pixels).save("pisica_de_mana.png")

In [ ]:
import IPython.display as ipd

ipd.display(ipd.Image("pisica_de_mana.png"))

## Scala de ghidare

Parametrul `guidance_scale` al pipeline-ului este scala $w$ din formula de mai sus, cu valoarea implicită 7,5. Codul de mai jos generează aceeași imagine, cu același zgomot de pornire, pentru patru valori ale scalei și le pune una lângă alta.

In [ ]:
import torch
from diffusers import StableDiffusionPipeline
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
dtype = torch.float16 if device != "cpu" else torch.float32
model_id = "stable-diffusion-v1-5/stable-diffusion-v1-5"
pipe = StableDiffusionPipeline.from_pretrained(model_id, dtype=dtype, variant="fp16").to(device)

prompt = "a watercolor painting of an orange cat sleeping on a park bench"
scales = [1.0, 3.0, 7.5, 15.0]
grid = Image.new("RGB", (len(scales) * 512, 512))
for column, w in enumerate(scales):
    generator = torch.Generator("cpu").manual_seed(0)
    image = pipe(prompt, guidance_scale=w, generator=generator).images[0]
    grid.paste(image, (512 * column, 0))
grid.save("scala_de_ghidare.png")

In [ ]:
import IPython.display as ipd

ipd.display(ipd.Image("scala_de_ghidare.png"))

## De la imagine la imagine

În generarea **de la imagine la imagine** (_image-to-image_), procesul invers nu pornește de la zgomot pur, ci de la o imagine existentă, trecută prin procesul direct până la un pas intermediar. Parametrul **`strength`** (intensitate) alege acest pas. Cu 50 de pași și o intensitate de 0,3, pipeline-ul adaugă zgomot cât pentru ultimii 15 pași și rulează doar acești 15 pași. Codul de mai jos pornește de la pisica generată mai sus, cu o descriere nouă, pentru trei valori ale intensității.

In [ ]:
import torch
from diffusers import StableDiffusionImg2ImgPipeline
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
dtype = torch.float16 if device != "cpu" else torch.float32
model_id = "stable-diffusion-v1-5/stable-diffusion-v1-5"
pipe = StableDiffusionImg2ImgPipeline.from_pretrained(model_id, dtype=dtype, variant="fp16").to(device)

init_image = Image.open("pisica_de_mana.png").convert("RGB")  # imaginea generată mai sus, fără pipeline
prompt = "an oil painting of an orange cat sleeping on a park bench, thick brush strokes"
strengths = [0.3, 0.6, 0.9]
grid = Image.new("RGB", ((len(strengths) + 1) * 512, 512))
grid.paste(init_image, (0, 0))
for column, strength in enumerate(strengths, start=1):
    generator = torch.Generator("cpu").manual_seed(0)
    image = pipe(prompt, image=init_image, strength=strength, generator=generator).images[0]
    grid.paste(image, (512 * column, 0))
grid.save("de_la_imagine_la_imagine.png")

In [ ]:
import IPython.display as ipd

ipd.display(ipd.Image("de_la_imagine_la_imagine.png"))